In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
datasets = {
    "APTOS": "/kaggle/input/datasets/sovitrath/diabetic-retinopathy-224x224-2019-data/colored_images",
    
    }

for name, path in datasets.items():
    print("\n" + "="*60)
    print(name)
    print("="*60)
    
    if not os.path.exists(path):
        print("Path does not exist")
        continue
    
    print("Path", path)
    
    for item in os.listdir(path)[:20]:
        print(" -", item)

import os
from collections import Counter

# =========================
# 1. APTOS
# =========================
aptos_path = (
    "/kaggle/input/datasets/sovitrath/"
    "diabetic-retinopathy-224x224-2019-data/"
    "colored_images"
)

print("=" * 60)
print("APTOS 2019")
print("=" * 60)

if os.path.exists(aptos_path):
    total = 0

    for cls in sorted(os.listdir(aptos_path)):
        cls_path = os.path.join(aptos_path, cls)

        if os.path.isdir(cls_path):
            images = [
                f for f in os.listdir(cls_path)
                if f.lower().endswith((".jpg", ".jpeg", ".png"))
            ]

            print(f"{cls}: {len(images)}")
            total += len(images)

    print("Total number of images:", total)
else:
    print("APTOS path does not exist.")

In [ ]:
import os
import pandas as pd
from PIL import Image

# ============================================================
# 1. Dataset path
# ============================================================

APTOS_PATH = (
    "/kaggle/input/datasets/sovitrath/"
    "diabetic-retinopathy-224x224-2019-data/"
    "colored_images"
)


# ============================================================
# 2. Helper functions
# ============================================================

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")


def get_image_files(folder):
    """Get all image files from the folder."""
    return [
        f for f in os.listdir(folder)
        if f.lower().endswith(IMAGE_EXTENSIONS)
    ]


def get_first_image_info(folder):
    """Get the format and dimensions of the first image."""
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.lower().endswith(IMAGE_EXTENSIONS):
                img_path = os.path.join(root, file)

                try:
                    with Image.open(img_path) as img:
                        return {
                            "format": img.format,
                            "size": f"{img.size[0]} x {img.size[1]}"
                        }
                except Exception:
                    continue

    return {
        "format": "Unknown",
        "size": "Unknown"
    }


# ============================================================
# 3. APTOS Statistics
# ============================================================

aptos_class_counts = {}
aptos_total = 0

if os.path.exists(APTOS_PATH):

    for cls in sorted(os.listdir(APTOS_PATH)):

        cls_path = os.path.join(APTOS_PATH, cls)

        if os.path.isdir(cls_path):

            images = get_image_files(cls_path)

            aptos_class_counts[cls] = len(images)

            aptos_total += len(images)

    aptos_info = get_first_image_info(APTOS_PATH)

else:

    aptos_info = {
        "format": "Path not found",
        "size": "Path not found"
    }

# ============================================================
# Create a dataset information table.
# ============================================================

summary_data = [

    {
        "Dataset": "APTOS 2019",
        "Modality": "Fundus",
        "Images": aptos_total,
        "Classes": len(aptos_class_counts),
        "Class distribution": str(aptos_class_counts),
        "Format": aptos_info["format"],
        "Example size": aptos_info["size"],
        "Path": APTOS_PATH
    }
]

summary_df = pd.DataFrame(summary_data)


# ============================================================
# Output results
# ============================================================

print("=" * 100)
print("DATASET SUMMARY")
print("=" * 100)

display(summary_df)


# ============================================================
# Output category details separately.
# ============================================================

print("\n" + "=" * 60)
print("APTOS 2019 CLASS DISTRIBUTION")
print("=" * 60)

display(
    pd.DataFrame(
        list(aptos_class_counts.items()),
        columns=["Class", "Images"]
    )
)

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    cohen_kappa_score
)

# =========================
# Basic Settings
# =========================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

In [ ]:
APTOS_PATH = (
    "/kaggle/input/datasets/sovitrath/"
    "diabetic-retinopathy-224x224-2019-data/"
    "colored_images"
)

# 统一成医学上常用的 0~4
CLASS_MAP = {
    "No_DR": 0,
    "Mild": 1,
    "Moderate": 2,
    "Severe": 3,
    "Proliferate_DR": 4
}

CLASS_NAMES = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative DR"
]

print(CLASS_MAP)

In [ ]:
import pandas as pd
import os
records = []

for class_folder, label in CLASS_MAP.items():

    folder = os.path.join(
        APTOS_PATH,
        class_folder
    )

    for filename in os.listdir(folder):

        if filename.lower().endswith(
            (".png", ".jpg", ".jpeg")
        ):

            records.append({
                "path": os.path.join(folder, filename),
                "label": label,
                "class_name": class_folder
            })

aptos_df = pd.DataFrame(records)

print("Total images:", len(aptos_df))
print()

print(
    aptos_df["label"]
    .value_counts()
    .sort_index()
)

In [ ]:
import os
import random
import numpy as np

# 1. Set a unified global random seed.
SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(SEED)
from sklearn.model_selection import train_test_split  # <-- 加上这行代码！
train_df, temp_df = train_test_split(
    aptos_df,
    test_size=0.30,
    random_state=SEED,
    stratify=aptos_df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
print("\nTrain distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation distribution:")
print(val_df["label"].value_counts().sort_index())

print("\nTest distribution:")
print(test_df["label"].value_counts().sort_index())

In [ ]:
import os
import cv2 as cv
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
class APTOSDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = Image.open(
            row["path"]
        ).convert("RGB")

        label = int(row["label"])

        if self.transform:
            image = self.transform(image)

        return image, label

import os
import random
import numpy as np
import pandas as pd
import cv2 as cv

# Machine Learning and Data Splitting
from sklearn.model_selection import train_test_split

# PyTorch Core and Data Pipelines
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Basic configuration
SEED = 42

import torch.nn as nn
from torchvision import models

# Do not load pre-trained weights (weights=None) to completely avoid network requests.
model = models.resnet18(weights=None)

# Modify the final layer to output five classes for the APTOS data.
model.fc = nn.Linear(model.fc.in_features, 5)

# Move to GPU or CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

train_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


val_test_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = APTOSDataset(
    train_df,
    transform=train_transform
)

val_dataset = APTOSDataset(
    val_df,
    transform=val_test_transform
)

test_dataset = APTOSDataset(
    test_df,
    transform=val_test_transform
)


BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1, 2, 3, 4])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print("Class weights:")
for i, w in enumerate(class_weights):
    print(
        CLASS_NAMES[i],
        ":",
        round(w.item(), 3)
    )

model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Modify the last layer.
model.fc = nn.Linear(
    model.fc.in_features,
    5
)

model = model.to(device)

print(model.fc)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer
):

    model.train()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item() * images.size(0)
        )

        preds = outputs.argmax(dim=1)

        all_preds.extend(
            preds.detach().cpu().numpy()
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )

    epoch_loss = (
        running_loss / len(loader.dataset)
    )

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    qwk = cohen_kappa_score(
        all_labels,
        all_preds,
        weights="quadratic"
    )

    return epoch_loss, accuracy, macro_f1, qwk

@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion
):

    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        running_loss += (
            loss.item() * images.size(0)
        )

        preds = outputs.argmax(dim=1)

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

    loss = (
        running_loss /
        len(loader.dataset)
    )

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    qwk = cohen_kappa_score(
        all_labels,
        all_preds,
        weights="quadratic"
    )

    return (
        loss,
        accuracy,
        macro_f1,
        qwk
    )

EPOCHS = 5

history = []

best_qwk = -1

for epoch in range(EPOCHS):

    train_loss, train_acc, train_f1, train_qwk = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    val_loss, val_acc, val_f1, val_qwk = evaluate(
        model,
        val_loader,
        criterion
    )

    scheduler.step(val_qwk)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}]"
    )

    print(
        f"Train | "
        f"Loss: {train_loss:.4f} | "
        f"Acc: {train_acc:.4f} | "
        f"F1: {train_f1:.4f} | "
        f"QWK: {train_qwk:.4f}"
    )

    print(
        f"Val   | "
        f"Loss: {val_loss:.4f} | "
        f"Acc: {val_acc:.4f} | "
        f"F1: {val_f1:.4f} | "
        f"QWK: {val_qwk:.4f}"
    )

    print("-" * 60)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_f1": train_f1,
        "train_qwk": train_qwk,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1,
        "val_qwk": val_qwk
    })

    # Save the best model
    if val_qwk > best_qwk:

        best_qwk = val_qwk

        torch.save(
            model.state_dict(),
            "/kaggle/working/best_aptos_resnet18.pth"
        )

        print("✓ Saved best model")

In [ ]:
model.load_state_dict(
    torch.load(
        "/kaggle/working/best_aptos_resnet18.pth",
        map_location=device
    )
)

test_loss, test_acc, test_f1, test_qwk = evaluate(
    model,
    test_loader,
    criterion
)

print("=" * 60)
print("APTOS TEST RESULTS")
print("=" * 60)

print(f"Accuracy : {test_acc:.4f}")
print(f"Macro F1 : {test_f1:.4f}")
print(f"QWK      : {test_qwk:.4f}")

In [ ]:
@torch.no_grad()
def get_predictions(model, loader):

    model.eval()

    preds = []
    labels = []

    for images, y in loader:

        images = images.to(device)

        outputs = model(images)

        p = outputs.argmax(dim=1)

        preds.extend(
            p.cpu().numpy()
        )

        labels.extend(
            y.numpy()
        )

    return np.array(labels), np.array(preds)


y_true, y_pred = get_predictions(
    model,
    test_loader
)

cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(7, 6))

plt.imshow(cm)

plt.xticks(
    range(5),
    CLASS_NAMES,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(5),
    CLASS_NAMES
)

plt.xlabel("Predicted")
plt.ylabel("True")

plt.title("APTOS 2019 Confusion Matrix")

for i in range(5):
    for j in range(5):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        digits=4
    )
)

In [ ]:
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    cohen_kappa_score,
    roc_auc_score
)

@torch.no_grad()
def get_predictions_and_probs(model, loader, device):
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    for images, labels in loader:

        images = images.to(device)

        outputs = model(images)

        # Convert to probability
        probs = torch.softmax(outputs, dim=1)

        # The class corresponding to the highest probability
        preds = torch.argmax(probs, dim=1)

        all_labels.extend(labels.numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    return (
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs)
    )

y_true, y_pred, y_prob = get_predictions_and_probs(
    model,
    test_loader,
    device
)

# Accuracy
accuracy = accuracy_score(y_true, y_pred)

# Macro F1
macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro"
)

# QWK
qwk = cohen_kappa_score(
    y_true,
    y_pred,
    weights="quadratic"
)

# macro AUROC
auroc_macro = roc_auc_score(
    y_true,
    y_prob,
    multi_class="ovr",
    average="macro"
)

auroc_weighted = roc_auc_score(
    y_true,
    y_prob,
    multi_class="ovr",
    average="weighted"
)

print("=" * 50)
print("APTOS TEST RESULTS")
print("=" * 50)

print(f"Accuracy       : {accuracy:.4f}")
print(f"Macro F1       : {macro_f1:.4f}")
print(f"QWK            : {qwk:.4f}")
print(f"Macro AUROC    : {auroc_macro:.4f}")
print(f"Weighted AUROC : {auroc_weighted:.4f}")

In [ ]:
from sklearn.preprocessing import label_binarize

# Convert labels to one-hot
y_true_bin = label_binarize(
    y_true,
    classes=[0, 1, 2, 3, 4]
)

print("=" * 50)
print("PER-CLASS AUROC")
print("=" * 50)

for i, class_name in enumerate(CLASS_NAMES):

    auc = roc_auc_score(
        y_true_bin[:, i],
        y_prob[:, i]
    )

    print(
        f"{class_name:20s}: {auc:.4f}"
    )

In [ ]:
!pip install grad-cam
import torch
import numpy as np
import matplotlib.pyplot as plt

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


# ==========================================
# 1. Load the best model based on QWK.
# ==========================================

model.load_state_dict(
    torch.load(
        "/kaggle/working/best_aptos_resnet18.pth",
        map_location=device
    )
)

model = model.to(device)
model.eval()


# ==========================================
# 2. APTOS Category Name
# ==========================================

classes = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative DR"
]


# ==========================================
# 3. Automatically identify correct and incorrect samples.
# ==========================================

correct_sample = None
wrong_sample = None

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probs = torch.softmax(outputs, dim=1)

        preds = outputs.argmax(dim=1)

        for i in range(images.size(0)):

            true_label = labels[i].item()
            pred_label = preds[i].item()

            confidence = probs[i, pred_label].item()

            # Find one with a correct prediction.
            if (
                correct_sample is None
                and pred_label == true_label
            ):

                correct_sample = (
                    images[i:i+1].detach().cpu(),
                    true_label,
                    pred_label,
                    confidence
                )

            # Find an example of an incorrect prediction.
            if (
                wrong_sample is None
                and pred_label != true_label
            ):

                wrong_sample = (
                    images[i:i+1].detach().cpu(),
                    true_label,
                    pred_label,
                    confidence
                )

            # Stop once both have been found.
            if (
                correct_sample is not None
                and wrong_sample is not None
            ):
                break

        if (
            correct_sample is not None
            and wrong_sample is not None
        ):
            break


print("Correct sample:")
print(
    f"True = {classes[correct_sample[1]]}, "
    f"Pred = {classes[correct_sample[2]]}, "
    f"Confidence = {correct_sample[3]:.2%}"
)

print()

print("Wrong sample:")
print(
    f"True = {classes[wrong_sample[1]]}, "
    f"Pred = {classes[wrong_sample[2]]}, "
    f"Confidence = {wrong_sample[3]:.2%}"
)

def generate_gradcam(sample):

    image_tensor = sample[0].to(device)

    true_label = sample[1]
    pred_label = sample[2]
    confidence = sample[3]

    # --------------------------
    # Denormalization
    # --------------------------

    mean = np.array(
        [0.485, 0.456, 0.406]
    )

    std = np.array(
        [0.229, 0.224, 0.225]
    )

    rgb_img = (
        image_tensor[0]
        .detach()
        .cpu()
        .numpy()
    )

    rgb_img = np.transpose(
        rgb_img,
        (1, 2, 0)
    )

    rgb_img = (
        rgb_img * std + mean
    )

    rgb_img = np.clip(
        rgb_img,
        0,
        1
    )

    # --------------------------
    # Grad-CAM
    # --------------------------

    target_layers = [
        model.layer4[-1]
    ]

    with GradCAM(
        model=model,
        target_layers=target_layers
    ) as cam:

        targets = [
            ClassifierOutputTarget(
                pred_label
            )
        ]

        grayscale_cam = cam(
            input_tensor=image_tensor,
            targets=targets
        )[0]

    # --------------------------
    # Overlaying heatmaps
    # --------------------------

    cam_image = show_cam_on_image(
        rgb_img,
        grayscale_cam,
        use_rgb=True
    )

    return (
        rgb_img,
        cam_image,
        true_label,
        pred_label,
        confidence
    )

correct_result = generate_gradcam(
    correct_sample
)

wrong_result = generate_gradcam(
    wrong_sample
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(12, 10)
)


# ==========================================
# Correct sample – Original image
# ==========================================

rgb_img = correct_result[0]

true_label = correct_result[2]
pred_label = correct_result[3]
confidence = correct_result[4]

axes[0, 0].imshow(rgb_img)

axes[0, 0].axis("off")

axes[0, 0].set_title(
    f"Correct Prediction - Original\n"
    f"True: {classes[true_label]}\n"
    f"Pred: {classes[pred_label]} "
    f"({confidence:.2%})"
)


# ==========================================
# Correct sample - Grad-CAM
# ==========================================

cam_image = correct_result[1]

axes[0, 1].imshow(cam_image)

axes[0, 1].axis("off")

axes[0, 1].set_title(
    f"Correct Prediction - Grad-CAM\n"
    f"Model focus for: {classes[pred_label]}"
)


# ==========================================
# Error Sample - Original Image
# ==========================================

rgb_img = wrong_result[0]

true_label = wrong_result[2]
pred_label = wrong_result[3]
confidence = wrong_result[4]

axes[1, 0].imshow(rgb_img)

axes[1, 0].axis("off")

axes[1, 0].set_title(
    f"Wrong Prediction - Original\n"
    f"True: {classes[true_label]}\n"
    f"Pred: {classes[pred_label]} "
    f"({confidence:.2%})"
)


# ==========================================
# Wrong example - Grad-CAM
# ==========================================

cam_image = wrong_result[1]

axes[1, 1].imshow(cam_image)

axes[1, 1].axis("off")

axes[1, 1].set_title(
    f"Wrong Prediction - Grad-CAM\n"
    f"Model focus for: {classes[pred_label]}"
)


plt.tight_layout()

plt.show()

In [ ]:
import os
import zipfile
import torch
import numpy as np
import matplotlib.pyplot as plt

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


# ==============================
# Configuration
# ==============================

NUM_IMAGES = 30

SAVE_DIR = "/kaggle/working/gradcam_results"

os.makedirs(SAVE_DIR, exist_ok=True)


# ==============================
# Load the best model
# ==============================

model.load_state_dict(
    torch.load(
        "/kaggle/working/best_aptos_resnet18.pth",
        map_location=device
    )
)

model = model.to(device)
model.eval()


classes = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative DR"
]

print("Model loaded.")

In [ ]:
# ==============================
# collect validation predictions
# ==============================

all_samples = []

model.eval()

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probs = torch.softmax(
            outputs,
            dim=1
        )

        preds = outputs.argmax(dim=1)

        for i in range(images.size(0)):

            all_samples.append({
                "image": images[i].detach().cpu(),
                "true_label": labels[i].item(),
                "pred_label": preds[i].item(),
                "confidence": probs[
                    i,
                    preds[i]
                ].item()
            })


print(
    f"Total validation images: {len(all_samples)}"
)

In [ ]:
# ==============================
# Group by category + Correct/Incorrect
# ==============================

groups = {}

for sample in all_samples:

    true_label = sample["true_label"]
    correct = (
        sample["true_label"]
        ==
        sample["pred_label"]
    )

    key = (
        true_label,
        correct
    )

    if key not in groups:
        groups[key] = []

    groups[key].append(sample)


# ==============================
# Sort each group by confidence.
# ==============================

for key in groups:

    groups[key].sort(
        key=lambda x: x["confidence"],
        reverse=True
    )


# ==============================
# Select samples as evenly as possible.
# ==============================

selected_samples = []

for true_label in range(5):

    # correct sample
    correct_group = groups.get(
        (true_label, True),
        []
    )

    # wrong sample
    wrong_group = groups.get(
        (true_label, False),
        []
    )

    # Take at most... for each ground-truth category
    # 3 right + 3 wrong
    selected_samples.extend(
        correct_group[:3]
    )

    selected_samples.extend(
        wrong_group[:3]
    )


# if over 30
selected_samples = selected_samples[:NUM_IMAGES]


print(
    f"Selected {len(selected_samples)} images"
)

for i, sample in enumerate(
    selected_samples
):

    print(
        f"{i+1:02d}. "
        f"True={classes[sample['true_label']]} | "
        f"Pred={classes[sample['pred_label']]} | "
        f"Conf={sample['confidence']:.2%}"
    )

In [ ]:
# ==============================
# Grad-CAM 
# ==============================

def generate_gradcam(sample):

    image_tensor = (
        sample["image"]
        .unsqueeze(0)
        .to(device)
    )

    true_label = sample["true_label"]
    pred_label = sample["pred_label"]
    confidence = sample["confidence"]


    # --------------------------
    # Denormalization
    # --------------------------

    mean = np.array([
        0.485,
        0.456,
        0.406
    ])

    std = np.array([
        0.229,
        0.224,
        0.225
    ])


    rgb_img = (
        image_tensor[0]
        .detach()
        .cpu()
        .numpy()
    )

    rgb_img = np.transpose(
        rgb_img,
        (1, 2, 0)
    )

    rgb_img = (
        rgb_img * std + mean
    )

    rgb_img = np.clip(
        rgb_img,
        0,
        1
    )


    # --------------------------
    # Grad-CAM
    # --------------------------

    target_layers = [
        model.layer4[-1]
    ]

    with GradCAM(
        model=model,
        target_layers=target_layers
    ) as cam:

        targets = [
            ClassifierOutputTarget(
                pred_label
            )
        ]

        grayscale_cam = cam(
            input_tensor=image_tensor,
            targets=targets
        )[0]


    # --------------------------
    # Superimpose
    # --------------------------

    cam_image = show_cam_on_image(
        rgb_img,
        grayscale_cam,
        use_rgb=True
    )


    return (
        rgb_img,
        cam_image,
        true_label,
        pred_label,
        confidence
    )

In [ ]:
# ==============================
# Batch generation 30 Grad-CAM
# ==============================

results = []

for idx, sample in enumerate(
    selected_samples
):

    print(
        f"[{idx+1}/{len(selected_samples)}] "
        f"Generating Grad-CAM..."
    )

    result = generate_gradcam(
        sample
    )

    results.append(result)


print("Finished!")

In [ ]:
# ==============================
# Save each comparison image.
# ==============================

for idx, result in enumerate(
    results
):

    rgb_img = result[0]
    cam_image = result[1]

    true_label = result[2]
    pred_label = result[3]
    confidence = result[4]


    fig, axes = plt.subplots(
        1,
        2,
        figsize=(10, 4)
    )


    # Original
    axes[0].imshow(rgb_img)
    axes[0].axis("off")

    axes[0].set_title(
        f"Original\n"
        f"True: {classes[true_label]}"
    )


    # Grad-CAM
    axes[1].imshow(cam_image)
    axes[1].axis("off")

    status = (
        "Correct"
        if true_label == pred_label
        else "Wrong"
    )

    axes[1].set_title(
        f"Grad-CAM ({status})\n"
        f"Pred: {classes[pred_label]} "
        f"({confidence:.2%})"
    )


    plt.tight_layout()


    filename = (
        f"{idx+1:02d}_"
        f"true_{true_label}_"
        f"pred_{pred_label}.png"
    )

    save_path = os.path.join(
        SAVE_DIR,
        filename
    )

    plt.savefig(
        save_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close(fig)


print(
    f"Saved to: {SAVE_DIR}"
)

In [ ]:
import shutil

#  gradcam_results ---> gradcam_results.zip
shutil.make_archive("gradcam_results", "zip", "gradcam_results")

print("Finish: gradcam_results.zip")